In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
filepath = "tree.json"

# Load the tree JSON file
with open(filepath, 'r') as f:
    tree = json.load(f)

In [ ]:
parent_map = {}
for node in tree["nodes"]:
    parent_map[node["ix"]] = node["parent_ix"]

In [ ]:
def get_path(ix):
    path = [ix]
    while not (path[-1] < 0):
        path.append(parent_map[path[-1]])
    return list(reversed(path[:-1]))


def cost_from_root(ix):
    return float(np.sum([tree["nodes"][jx]["cost"] for jx in get_path(ix)]))


def msec(x):
    return (1 / np.cos(0.5 * x)) - 1


def distance(x, y):
    xy_cost = np.linalg.norm(x[0:2] - y[0:2])
    dyaw = np.clip(x[2] - y[2], -np.pi, np.pi)
    yaw_cost = 10 * msec(dyaw)
    velocity_cost = 0.5 * np.abs(x[3] - y[3])
    return float(xy_cost + yaw_cost + velocity_cost)
    # return float(xy_cost)

In [ ]:
goal = np.array([20.0, 0.0, 0.0, 0.0])


ixs = [node["ix"] for node in tree["nodes"]]
ds = np.array([distance(np.array(node["state"]), goal) for node in tree["nodes"]])


max_dist_for_solve = 0.5

near_goal_ixs = np.argwhere(ds < max_dist_for_solve).flatten()

import warnings
if len(near_goal_ixs) == 0:
    min_dist = np.min(ds)
    warnings.warn(f"Planning problem not solved (min-dist-to-goal = {min_dist:.3f}), choosing best_ix as nearest-to-goal")
    best_ix = int(ixs[np.argmin(ds)])
else:
    cost_from_root__near_goal_ixs = np.array([cost_from_root(int(ix)) for ix in near_goal_ixs])
    best_ix = int(near_goal_ixs[int(np.argmin(cost_from_root__near_goal_ixs))])

path = get_path(best_ix)


In [ ]:
def plot_tree(tree):
    """
    Plot the tree from a JSON file by drawing lines connecting each state in the trajectory of each node.
    
    The JSON is expected to contain a "nodes" list where each node has:
      - "ix": an integer node index.
      - "state": a list representing the node state (first two entries are x and y).
      - "state_sequence": a list of state arrays (each with at least two numbers for x and y).
    """
   
    fig, ax = plt.subplots(figsize=(20, 4))
    
    for node in tree.get("nodes", []):
        if node["ix"] in path:
            color = "tab:blue"
            lw = 3
            alpha = 1.0
            marker = "x"
            markersize = 7
            zorder = 100
        elif node["parent_ix"] in path:
            color = "k"
            lw = 1
            alpha = 0.5
            marker = None
            markersize = 5
            zorder = 90
        else:
            color = "tab:grey"
            lw = 0.5
            alpha = 0.3
            marker = None
            markersize = 3
            zorder = 80

        # Plot the node's main state as a marker
        x, y = node["state"][0], node["state"][1]
        ax.plot(x, y, color=color, marker="o", markersize=markersize, alpha=alpha, zorder=zorder)
        # ax.text(x, y, f'{node["ix"]}', fontsize=9, color=color, ha='right', va='bottom')
        
        # Plot the trajectory for this node, if available.
        # Each element in state_sequence is expected to be a state vector [x, y, ...].
        state_seq = node.get("state_sequence", [])
        if state_seq:
            # Extract x and y coordinates from the state sequence.
            x_vals = [state[0] for state in state_seq]
            y_vals = [state[1] for state in state_seq]

            ax.plot(x_vals, y_vals, color=color, marker=marker, linewidth=lw, alpha=alpha, zorder=zorder)
            # Optionally, mark the start and end of the trajectory.
            # ax.plot(x_vals[0], y_vals[0], 'go', label='Start' if node["ix"] == tree["nodes"][0]["ix"] else "")
            # ax.plot(x_vals[-1], y_vals[-1], 'ro', label='End' if node["ix"] == tree["nodes"][0]["ix"] else "")
    circle = plt.Circle((10, 0), 1, color='tab:red', fill=True)
    ax.add_patch(circle)

    # ax.scatter(*(0, 0), label="Start", c="tab:blue", marker="^", s=100)
    ax.scatter(*(20, 0), label="Goal", c="tab:green", marker="*", s=400)



    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_title("Kinodynamic Tree")
    ax.grid(True)
    ax.axis("equal")
    ax.set_xlim((-1, 21))
    ax.set_ylim((-0.1, 2.1))
    ax.legend()
    return fig, ax


In [ ]:
# costs = node["cost"] for node in tree["nodes"]

In [ ]:
cost_from_root(best_ix)

In [ ]:
from pathlib import Path
import pandas as pd
traj_x_df = pd.read_csv(next(Path("data/state").glob("*.csv")), header=None)
traj_x_df.columns = ["x", "y", "yaw", "v"]
traj_x_df

In [ ]:
traj_u_df = pd.read_csv(next(Path("data/action").glob("*.csv")), header=None)
traj_u_df.columns = ["a", "k"]
traj_u_df

In [ ]:
import plotly.express as px
px.line(traj_u_df)

In [ ]:
fig, ax = plot_tree(tree)

ax.plot(traj_x_df.x, traj_x_df.y, lw=5, marker="|", markersize=10, c="tab:purple", zorder=1000)
plt.show()

In [ ]:
fig, axs = plt.subplots(nrows=2, sharex=True, figsize=(20, 4))

t_now = 0.0
for node in tree.get("nodes", []):
    # Root - skip
    if node["ix"] == 0:
        continue

    if node["ix"] in path:
        action_seq = node.get("action_sequence", [])
        if action_seq:
            total_time = node.get("total_time")
            t = np.linspace(t_now, t_now + total_time, len(action_seq))
            t_now += total_time

            accl = [action[0] for action in action_seq]
            curv = [action[1] for action in action_seq]
            axs[0].plot(t, accl, c="k", marker='|')
            axs[1].plot(t, curv, c="k", marker='|')

for ax in axs:
    ax.grid()
fig.tight_layout()

In [ ]:
fig, axs = plt.subplots(nrows=2, sharex=True, figsize=(20, 4))

t_now = 0.0
for node in tree.get("nodes", []):
    # Root - skip
    if node["ix"] == 0:
        continue

    if node["ix"] in path:
        state_seq = node.get("state_sequence", [])
        if state_seq:
            total_time = node.get("total_time")
            t = np.linspace(t_now, t_now + total_time, len(action_seq))
            t_now += total_time

            yaw = [state[2] for state in state_seq[:-1]]
            v = [state[3] for state in state_seq[:-1]]
            axs[0].plot(t, yaw, c="k", marker='|')
            axs[1].plot(t, v, c="k", marker='|')

for ax in axs:
    ax.grid()
fig.tight_layout()